In [4]:
import pandas as pd
import numpy as np
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)

Customers: (60000, 9)
Orders: (200000, 14)
Products: (10000, 8)


In [3]:
customers.head()

,customer_id,customer_name,age,gender,city,state,customer_segment,signup_date,preferred_channel
0,CUST000001,Customer_000001,27,Female,Pune,Maharashtra,Premium,2023-08-06,Website
1,CUST000002,Customer_000002,45,Male,Hyderabad,Telangana,Regular,2025-06-24,Website
2,CUST000003,Customer_000003,53,Female,Surat,Gujarat,Premium,2025-02-22,Mobile App
3,CUST000004,Customer_000004,27,Male,Bhopal,Madhya Pradesh,Regular,2023-08-17,Website
4,CUST000005,Customer_000005,31,Male,Noida,Uttar Pradesh,New,2023-02-07,Website


# For missing values

In [4]:
customers.isnull().sum()

customer_id          0
customer_name        0
age                  0
gender               0
city                 0
state                0
customer_segment     0
signup_date          0
preferred_channel    0
dtype: int64

In [5]:
orders.head()

,order_id,customer_id,product_id,order_date,order_status,quantity,unit_price,discount_pct,discount_amount,freight_amount,total_amount,payment_method,delivery_date,estimated_delivery_date
0,ORD0000001,CUST043981,PROD02971,2024-03-04,Delivered,3,255.56,15,115.00,193.45,845.13,Debit Card,2024-03-14,2024-03-12
1,ORD0000002,CUST011019,PROD06966,2023-07-09,Delivered,1,277.63,0,0.00,391.43,669.06,Debit Card,2023-07-13,2023-07-17
2,ORD0000003,CUST007980,PROD00423,2023-02-22,Delivered,1,1207.13,15,181.07,386.09,1412.15,UPI,2023-02-25,2023-03-01
3,ORD0000004,CUST014065,PROD02332,2025-12-16,Delivered,1,256.62,0,0.00,190.21,446.83,Debit Card,2025-12-20,2025-12-23
4,ORD0000005,CUST017410,PROD02877,2024-08-27,Delivered,1,788.24,0,0.00,283.30,1071.54,UPI,2024-08-29,2024-09-03


In [6]:
orders.isnull().sum()

order_id                   0
customer_id                0
product_id                 0
order_date                 0
order_status               0
quantity                   0
unit_price                 0
discount_pct               0
discount_amount            0
freight_amount             0
total_amount               0
payment_method             0
delivery_date              0
estimated_delivery_date    0
dtype: int64

In [7]:
products.head()

,product_id,product_name,category,sub_category,brand,unit_price,stock_quantity,rating
0,PROD00001,Product_00001,Sports,Fitness Equipment,Orion,446.04,266,4.5
1,PROD00002,Product_00002,Electronics,Laptop,Vertex,594.89,436,3.8
2,PROD00003,Product_00003,Books,Non-Fiction,UrbanX,814.08,229,4.9
3,PROD00004,Product_00004,Fashion,Jeans,PrimeTech,254.57,291,4.7
4,PROD00005,Product_00005,Home,Chair,Vertex,350.36,451,4.5


In [8]:
products.isnull().sum()

product_id        0
product_name      0
category          0
sub_category      0
brand             0
unit_price        0
stock_quantity    0
rating            0
dtype: int64

# Finding Duplicates

In [9]:
customers.duplicated().sum()

0

In [10]:
orders.duplicated().sum()

0

In [11]:
products.duplicated().sum()

0

In [12]:
customers["customer_id"].duplicated().sum()

0

so no dupliactes values present on 3 tables 


# Showing realtion between customers and orders

In [14]:
orders["customer_id"].isin(customers["customer_id"]).value_counts()

customer_id
True    200000
Name: count, dtype: int64

# Product and order relationship

In [15]:
orders["product_id"].isin(products["product_id"]).value_counts()

product_id
True    200000
Name: count, dtype: int64

# Order Table is fact table Because the Orders table is related to both Customers and Products

## Case Study 1 — Declining Customer Revenue

## Business situation:
The company's management has noticed that revenue growth has slowed down, even though the company continues to acquire customers.

## The Managers wants to understand:
Why are customers not generating enough repeat revenue, and which customer groups and products are responsible?

To solve these problems we have to take some stpes 




## Step1: Connect the three tables

In [25]:
import pandas as pd

customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")

analysis_df = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
)

print(analysis_df.shape)
print(analysis_df.head())

Now analysis_df contains customer, order and product information.

## Step 2 — Revenue by customer segment

The first question management wants answered is:
Which customer segment generates the most revenue?

In [28]:
segment_analysis = (
    analysis_df
    .groupby("customer_segment")
    .agg(
        customers=("customer_id", "nunique"),
        orders=("order_id", "nunique"),
        revenue=("total_amount", "sum")
    )
)

print(segment_analysis)

                  customers  orders       revenue
customer_segment                                 
New                   19986   69165  9.708066e+07
Premium                8762   30368  4.244023e+07
Regular               29172  100467  1.403959e+08


Initial insight
Regular customers generate the highest total revenue, approximately ₹140.40 million. However, we should not immediately conclude that Regular customers are the most valuable.
Why?
Because there are far more Regular customers than Premium customers.
So we need to go deeper.

# Step 3 — Revenue per customer

In [29]:
segment_analysis["revenue_per_customer"] = (
    segment_analysis["revenue"] /
    segment_analysis["customers"]
)

print(segment_analysis)

                  customers  orders       revenue  revenue_per_customer
customer_segment                                                       
New                   19986   69165  9.708066e+07           4857.433334
Premium                8762   30368  4.244023e+07           4843.669692
Regular               29172  100467  1.403959e+08           4812.692835


Important insight
The differences are actually very small.
So the business isn't simply a case of:
"Premium customers spend much more."

In this dataset, the three segments have relatively similar revenue per customer.
That's an important finding.

# Step 4 — Purchase frequency

Now management wants to know:
Are customers purchasing frequently enough?

In [30]:
segment_analysis["orders_per_customer"] = (
    segment_analysis["orders"] /
    segment_analysis["customers"]
)

print(
    segment_analysis[
        ["orders_per_customer"]
    ]
)

                  orders_per_customer
customer_segment                     
New                          3.460672
Premium                      3.465875
Regular                      3.443953


Again, the difference is very small.
So customer purchasing frequency isn't dramatically different between segments.

# Step 5 — Check repeat customers

Now we investigate customer retention.

In [31]:
customer_orders = (
    orders
    .groupby("customer_id")["order_id"]
    .nunique()
)

one_time = (customer_orders == 1).sum()
repeat = (customer_orders > 1).sum()

total = len(customer_orders)

print("One-time customers:", one_time)
print("Repeat customers:", repeat)
print("Repeat customer rate:", repeat / total * 100)

One-time customers: 7191
Repeat customers: 50729
Repeat customer rate: 87.58459944751381


Business insight
This is actually very strong customer repeat behavior.
Approximately:
87.6% of customers are repeat customers.

Therefore, based on this dataset, we cannot claim that poor customer retention is the main reason for weak revenue growth.
That's exactly why proper analysis is important—you don't force the data to support the original business assumption.

# Step 6 — What are customers buying?

In [32]:
segment_category = (
    analysis_df
    .groupby(["customer_segment", "category"])
    .agg(
        orders=("order_id", "nunique"),
        revenue=("total_amount", "sum")
    )
    .reset_index()
)

print(segment_category)

   customer_segment     category  orders      revenue
0               New       Beauty   11062  16013927.84
1               New        Books   11554  16524075.02
2               New  Electronics   11216  15342221.29
3               New      Fashion   11689  16272749.88
4               New         Home   11618  15783638.61
5               New       Sports   12026  17144049.97
6           Premium       Beauty    4982   7027880.87
7           Premium        Books    4969   7143615.26
8           Premium  Electronics    4915   6631251.58
9           Premium      Fashion    5011   6882409.27
10          Premium         Home    5214   7152706.22
11          Premium       Sports    5277   7602370.64
12          Regular       Beauty   16206  22985576.02
13          Regular        Books   16796  24106032.54
14          Regular  Electronics   16141  21857388.18
15          Regular      Fashion   16769  23149069.97
16          Regular         Home   17221  23532875.26
17          Regular       Sp

# Step 7 — Business conclusion
Now we can give management an actual answer.
## Finding 1
Regular customers are the largest revenue contributor, generating approximately ₹140.4M.
## Finding 2
Premium customers are a much smaller group:
8,762 customers
but their revenue per customer is approximately:
₹4,844
which is similar to New and Regular customers.
## Finding 3
Customer retention does not appear to be a major problem in this dataset.
The repeat-customer rate is approximately:
87.6%
## Finding 4
Sports is the strongest revenue category across all three customer segments.

# Final Business Solution
## Based on the analysis:
The company should not focus primarily on increasing customer retention, because the existing repeat-customer rate is already high. Instead, the stronger opportunity is to increase revenue per existing customer and optimize category-specific marketing.

# Business Recommendation
Shift the strategy from broad customer-retention initiatives toward increasing revenue per existing customer through personalized cross-selling, upselling and category-specific campaigns, particularly around high-performing categories.

Premium customers demonstrate the highest retention, with approximately 88.1% making repeat purchases. This indicates that premium customers are the most loyal segment and should be prioritized for retention and loyalty programs.”

# Case Study 2 — Customer Revenue & Retention Analysis
## Business problem
The e-commerce company has 60,000 customers and 200,000 orders.
Management wants to know:
Who are our most valuable customers, how many customers come back to purchase again, and which customer segments should we target to increase revenue?

This is a very realistic Data Analyst case study.

# Prepare the dates

customers['signup_date'] = pd.to_datetime(
    customers['signup_date']
)

orders['order_date'] = pd.to_datetime(
    orders['order_date']
)

orders['delivery_date'] = pd.to_datetime(
    orders['delivery_date']
)

orders['estimated_delivery_date'] = pd.to_datetime(
    orders['estimated_delivery_date']
)

## Why?
Because later we need to calculate things such as:
- Monthly revenue
- First purchase date
- Last purchase date
- Customer lifetime
- Days between purchases
- Retention/cohort analysis

# Step 2 — Understand order status

In [7]:
orders['order_status'].value_counts()

order_status
Delivered     155934
Shipped        16105
Processing     11882
Cancelled       9962
Returned        6117
Name: count, dtype: int64

Why do we need this?
Because not every order should be treated as realized revenue.
For our customer revenue analysis, we'll use:
Delivered orders = completed customer purchases

In [8]:
# Create a separate dataset:
delivered_orders = orders[
    orders['order_status'] == 'Delivered'
].copy()

In [9]:
delivered_orders.shape

(155934, 14)

# Step 3 — Calculate the overall business KPIs

In [10]:
# Total Customers
total_customers = customers['customer_id'].nunique()

print("Total Customers:", total_customers)

Total Customers: 60000


In [11]:
# Total orders
total_orders = orders['order_id'].nunique()

print("Total Orders:", total_orders)

Total Orders: 200000


In [12]:
# Delivered orders
delivered_orders_count = (
    delivered_orders['order_id'].nunique()
)

print("Delivered Orders:", delivered_orders_count)

Delivered Orders: 155934


In [13]:
# delivered revenue
delivered_revenue = delivered_orders['total_amount'].sum()

print(f"Delivered Revenue: ₹{delivered_revenue:,.2f}")

Delivered Revenue: ₹218,133,215.14


In [14]:
# Average order value
aov = delivered_revenue / delivered_orders_count

print(f"AOV: ₹{aov:,.2f}")

AOV: ₹1,398.88


# Step 4 — Calculate customer-level metrics
This is the most important transformation in this case study.
Your orders table contains many rows for the same customer.

In [15]:
customer_analysis = (
    delivered_orders
    .groupby('customer_id')
    .agg(
        total_orders=('order_id', 'nunique'),
        total_quantity=('quantity', 'sum'),
        total_revenue=('total_amount', 'sum'),
        average_order_value=('total_amount', 'mean'),
        average_discount=('discount_pct', 'mean'),
        first_order_date=('order_date', 'min'),
        last_order_date=('order_date', 'max')
    )
    .reset_index()
)
customer_analysis.head()

,customer_id,total_orders,total_quantity,total_revenue,average_order_value,average_discount,first_order_date,last_order_date
0,CUST000001,2,6,2962.01,1481.005,20.000000,2024-01-16,2025-01-21
1,CUST000002,1,3,1878.46,1878.460,0.000000,2025-08-23,2025-08-23
2,CUST000003,2,2,1047.64,523.820,10.000000,2024-05-28,2024-10-31
3,CUST000004,4,9,8199.16,2049.790,12.500000,2023-06-04,2025-11-17
4,CUST000005,3,6,2098.08,699.360,8.333333,2024-04-21,2025-06-29


In [16]:
customer_analysis.shape

(55578, 8)

# Step 5 — Calculate customer lifetime

In [17]:
customer_analysis['customer_lifetime_days'] = (
    customer_analysis['last_order_date']
    - customer_analysis['first_order_date']
).dt.days

In [18]:
customer_analysis[
    ['customer_id',
     'first_order_date',
     'last_order_date',
     'customer_lifetime_days']
].head()

,customer_id,first_order_date,last_order_date,customer_lifetime_days
0,CUST000001,2024-01-16,2025-01-21,371
1,CUST000002,2025-08-23,2025-08-23,0
2,CUST000003,2024-05-28,2024-10-31,156
3,CUST000004,2023-06-04,2025-11-17,897
4,CUST000005,2024-04-21,2025-06-29,434


# Business question
Do customers with longer purchasing relationships generate more revenue?

We'll investigate this later.

# Step 6 — Classify customers as One-Time or Repeat

In [19]:
customer_analysis['customer_type'] = np.where(
    customer_analysis['total_orders'] == 1,
    'One-Time',
    'Repeat'
)

In [20]:
customer_analysis['customer_type'].value_counts()

customer_type
Repeat      43931
One-Time    11647
Name: count, dtype: int64

# Step 7 — Compare One-Time vs Repeat Customers
Now aggregate:

In [21]:
customer_type_analysis = (
    customer_analysis
    .groupby('customer_type')
    .agg(
        customers=('customer_id', 'nunique'),
        total_revenue=('total_revenue', 'sum'),
        average_revenue=('total_revenue', 'mean'),
        average_orders=('total_orders', 'mean'),
        average_aov=('average_order_value', 'mean')
    )
    .reset_index()
)
customer_type_analysis

,customer_type,customers,total_revenue,average_revenue,average_orders,average_aov
0,One-Time,11647,1.631177e+07,1400.512227,1.000000,1400.512227
1,Repeat,43931,2.018214e+08,4594.055433,3.284401,1398.323875


Business questions
Now we can answer:
How many customers are repeat buyers?

How much revenue comes from repeat customers?

Are repeat customers more valuable?

# Step 8 — Calculate Repeat Customer Rate

In [22]:
repeat_customers = (
    customer_analysis[
        customer_analysis['customer_type'] == 'Repeat'
    ]['customer_id'].nunique()
)

customers_with_orders = customer_analysis['customer_id'].nunique()

repeat_rate = (
    repeat_customers / customers_with_orders
) * 100

print(f"Repeat Customer Rate: {repeat_rate:.2f}%")

Repeat Customer Rate: 79.04%


Business interpretation
For example, if the result is 79%:
Approximately 79% of customers who made at least one delivered purchase returned to purchase again.

That is a strong retention indicator.

# Step 9 — Merge customer information
Now we need demographic and customer attributes.
Take the relevant columns:

In [24]:
customer_info = customers[
    [
        'customer_id',
        'age',
        'gender',
        'city',
        'state',
        'customer_segment',
        'preferred_channel'
    ]
]

In [25]:
customer_analysis = customer_analysis.merge(
    customer_info,
    on='customer_id',
    how='left'
)

In [26]:
customer_analysis.head()

,customer_id,total_orders,total_quantity,total_revenue,average_order_value,average_discount,first_order_date,last_order_date,customer_lifetime_days,customer_type,age,gender,city,state,customer_segment,preferred_channel
0,CUST000001,2,6,2962.01,1481.005,20.000000,2024-01-16,2025-01-21,371,Repeat,27,Female,Pune,Maharashtra,Premium,Website
1,CUST000002,1,3,1878.46,1878.460,0.000000,2025-08-23,2025-08-23,0,One-Time,45,Male,Hyderabad,Telangana,Regular,Website
2,CUST000003,2,2,1047.64,523.820,10.000000,2024-05-28,2024-10-31,156,Repeat,53,Female,Surat,Gujarat,Premium,Mobile App
3,CUST000004,4,9,8199.16,2049.790,12.500000,2023-06-04,2025-11-17,897,Repeat,27,Male,Bhopal,Madhya Pradesh,Regular,Website
4,CUST000005,3,6,2098.08,699.360,8.333333,2024-04-21,2025-06-29,434,Repeat,31,Male,Noida,Uttar Pradesh,New,Website


Now your analytical dataset contains both:
Purchasing behavior
total_orders
total_quantity
total_revenue
average_order_value
average_discount

and:
Customer characteristics
age
gender
state
customer_segment
preferred_channel

# Step 10 — Analyze customer segments

In [27]:
segment_analysis = (
    customer_analysis
    .groupby('customer_segment')
    .agg(
        customers=('customer_id', 'nunique'),
        total_orders=('total_orders', 'sum'),
        total_revenue=('total_revenue', 'sum'),
        average_revenue=('total_revenue', 'mean'),
        average_aov=('average_order_value', 'mean')
    )
    .reset_index()
)
segment_analysis

,customer_segment,customers,total_orders,total_revenue,average_revenue,average_aov
0,New,19181,53885,7.569514e+07,3946.360435,1402.150867
1,Premium,8400,23623,3.300879e+07,3929.618039,1401.246189
2,Regular,27997,78426,1.094293e+08,3908.607498,1395.735553


# Business question
## Which customer segment generates the most revenue?

But don't stop there.
We also want:
## Which segment generates the most revenue per customer?

That's why we calculate average_revenue.

# Step 11 — Repeat rate by customer segment

In [28]:
customer_analysis['is_repeat'] = np.where(
    customer_analysis['total_orders'] > 1,
    1,
    0
)

In [29]:
segment_retention = (
    customer_analysis
    .groupby('customer_segment')
    .agg(
        customers=('customer_id', 'nunique'),
        repeat_customers=('is_repeat', 'sum')
    )
    .reset_index()
)

In [30]:
# Calculate the percentage:
segment_retention['repeat_rate'] = (
    segment_retention['repeat_customers']
    / segment_retention['customers']
) * 100
segment_retention

,customer_segment,customers,repeat_customers,repeat_rate
0,New,19181,15226,79.380637
1,Premium,8400,6692,79.666667
2,Regular,27997,22013,78.626281


Business question
Which customer segment has the highest repeat-purchase rate?

# Step 12 — Analyze preferred channel

In [32]:
channel_analysis = (
    customer_analysis
    .groupby('preferred_channel')
    .agg(
        customers=('customer_id', 'nunique'),
        total_orders=('total_orders', 'sum'),
        total_revenue=('total_revenue', 'sum'),
        average_revenue=('total_revenue', 'mean'),
        average_aov=('average_order_value', 'mean'),
        repeat_rate=('is_repeat', 'mean')
    )
    .reset_index()
)

In [33]:
channel_analysis['repeat_rate'] *= 100

In [34]:
channel_analysis

,preferred_channel,customers,total_orders,total_revenue,average_revenue,average_aov,repeat_rate
0,Mobile App,25138,70590,9.930264e+07,3950.299770,1408.451478,78.912404
1,Website,30440,85344,1.188306e+08,3903.764111,1390.797595,79.152431


Business question
Does the preferred purchasing channel influence customer value and repeat behavior?

# Step 13 — Analyze customer revenue drivers

Now we start diagnostic analysis.
We want to know what is related to revenue.

In [35]:
numeric_columns = [
    'age',
    'total_orders',
    'total_quantity',
    'total_revenue',
    'average_order_value',
    'average_discount',
    'customer_lifetime_days'
]

customer_analysis[numeric_columns].corr()

,age,total_orders,total_quantity,total_revenue,average_order_value,average_discount,customer_lifetime_days
age,1.000000,0.004144,0.001975,-0.000689,-0.003216,-0.003238,0.009099
total_orders,0.004144,1.000000,0.838234,0.682402,0.000204,0.000039,0.710370
total_quantity,0.001975,0.838234,1.000000,0.764193,0.228314,0.001757,0.594931
total_revenue,-0.000689,0.682402,0.764193,1.000000,0.626051,-0.039883,0.482730
average_order_value,-0.003216,0.000204,0.228314,0.626051,1.000000,-0.068202,-0.002457
average_discount,-0.003238,0.000039,0.001757,-0.039883,-0.068202,1.000000,-0.001149
customer_lifetime_days,0.009099,0.710370,0.594931,0.482730,-0.002457,-0.001149,1.000000


Business question
Which numerical factors have the strongest relationship with customer revenue?

# Step 14 — Monthly revenue trend

In [36]:
delivered_orders['order_month'] = (
    delivered_orders['order_date']
    .dt.to_period('M')
)

In [37]:
monthly_revenue = (
    delivered_orders
    .groupby('order_month')
    .agg(
        revenue=('total_amount', 'sum'),
        orders=('order_id', 'nunique')
    )
    .reset_index()
)

In [38]:
monthly_revenue['order_month'] = (
    monthly_revenue['order_month']
    .dt.to_timestamp()
)

In [39]:
monthly_revenue

,order_month,revenue,orders
0,2023-01-01,6153790.41,4434
1,2023-02-01,5710580.96,4079
2,2023-03-01,6049163.33,4430
3,2023-04-01,6048494.75,4237
4,2023-05-01,6096773.76,4446
5,2023-06-01,6015076.61,4248
6,2023-07-01,6224828.60,4433
7,2023-08-01,6060033.74,4361
8,2023-09-01,5958596.69,4210
9,2023-10-01,6293029.45,4452


# Key Insights
Overall KPIs: 60K customers, 200K orders, 155,934 delivered orders, and ₹218.13M delivered revenue.

Order Status: Delivered orders form the majority; cancellations and returns represent revenue leakage.

Customer Value: Average delivered order value is approximately ₹1,399.

Customer Revenue: Repeat customers contribute the majority of total delivered revenue.

Customer Lifetime: Longer customer relationships are associated with higher revenue.

One-Time vs Repeat: 79.05% of purchasing customers are repeat customers.

Repeat Customer Value: Repeat customers generate about 3.3× more revenue per customer than one-time customers.

Retention: Customer retention is a major driver of overall revenue.

Customer Attributes: Demographics alone do not strongly explain customer revenue.

Customer Segments: Regular customers generate the highest total revenue because they are the largest segment.

Segment Retention: Premium customers have the highest repeat rate, but the difference is small.

Channel: Website and Mobile App perform similarly; the Mobile App has slightly higher AOV.

Revenue Drivers: Quantity, order frequency, and AOV have the strongest relationships with customer revenue.

Monthly Revenue: Monthly analysis helps identify revenue trends, peaks, low-performing periods, and seasonal patterns.